In [ ]:
# Cell 1 — Fresh Load
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv('content/telco_churn.csv')
df.head()

In [ ]:
# Cell 2 — Fix Data Types
# TotalCharges is object; convert to numeric (11 rows have spaces → NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('TotalCharges nulls after conversion:', df['TotalCharges'].isnull().sum())

In [ ]:
# Cell 3 — Handle Missing Values
mean_tc = df['TotalCharges'].mean()
df['TotalCharges'] = df['TotalCharges'].fillna(mean_tc)  # correct pandas 3.0 syntax
print('Nulls remaining:', df.isna().sum().sum())

In [ ]:
# Cell 4 — Drop customerID (not a feature)
df = df.drop(columns=['customerID'])

In [ ]:
# Cell 5 — Encode Target Variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
# Cell 6 — Clean 'No internet/phone service' values
# Collapse these into plain 'No' — semantically they mean the same thing
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in service_cols:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

In [ ]:
# Cell 7 — One-Hot Encode nominal categoricals
df = pd.get_dummies(df, columns=['InternetService', 'PaymentMethod',
                                  'OnlineSecurity', 'MultipleLines'],
                    drop_first=True)

In [ ]:
# Cell 8 — Ordinal Encode Contract (has natural order)
contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df['Contract'] = df['Contract'].map(contract_map)
print('Contract nulls after mapping:', df['Contract'].isnull().sum())  # should be 0

In [ ]:
# Cell 9 — Label Encode remaining binary object columns
le = LabelEncoder()
binary_obj_cols = [col for col in df.columns
                   if df[col].dtype == 'object' and col != 'Churn']
for col in binary_obj_cols:
    df[col] = le.fit_transform(df[col])
print('Remaining object columns:', binary_obj_cols)

In [ ]:
# Cell 10 — Convert bool columns → 0/1
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
# Cell 11 — Feature Engineering
df['avg_monthly_spend'] = df['TotalCharges'] / (df['tenure'] + 1)  # avoids div/0
df['is_long_term'] = (df['tenure'] >= 24).astype(int)

In [ ]:
# Cell 12 — Train/Test Split
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

In [ ]:
# Cell 13 — Save clean data
# FIX: index=False prevents the 'Unnamed: 0' ghost column on reload
df.to_csv('telco_churn_clean.csv', index=False)
print('Saved clean CSV — no Unnamed: 0 column on reload.')

In [ ]:
# Cell 14 — VIF Round 1 (all numeric features)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns

# FIX: collist reset each time so it doesn't accumulate duplicates across runs
col_list = []
for i in X_train.columns:
    if X_train[i].dtype != 'object':
        col_list.append(i)

z = X_train[col_list]
vif_df = pd.DataFrame()
vif_df['Features'] = z.columns
vif_df['VIF'] = [variance_inflation_factor(z.values, i) for i in range(z.shape[1])]
vif_df

In [ ]:
# Cell 15 — Drop MonthlyCharges (highest VIF ~236)
X_train.drop('MonthlyCharges', axis=1, inplace=True)
X_test.drop('MonthlyCharges', axis=1, inplace=True)  # FIX: mirror drop on X_test

In [ ]:
# Cell 16 — VIF Round 2
col_list = []  # FIX: reset collist
for i in X_train.columns:
    if X_train[i].dtype != 'object':
        col_list.append(i)

z = X_train[col_list]
vif_df = pd.DataFrame()
vif_df['Features'] = z.columns
vif_df['VIF'] = [variance_inflation_factor(z.values, i) for i in range(z.shape[1])]
vif_df

In [ ]:
# Cell 17 — Drop tenure (VIF ~29 after previous drop)
X_train.drop('tenure', axis=1, inplace=True)
X_test.drop('tenure', axis=1, inplace=True)  # FIX: mirror drop on X_test

In [ ]:
# Cell 18 — VIF Round 3
col_list = []  # FIX: reset collist
for i in X_train.columns:
    if X_train[i].dtype != 'object':
        col_list.append(i)

z = X_train[col_list]
vif_df = pd.DataFrame()
vif_df['Features'] = z.columns
vif_df['VIF'] = [variance_inflation_factor(z.values, i) for i in range(z.shape[1])]
vif_df

In [ ]:
# Cell 19 — Drop TotalCharges (VIF ~10.7)
X_train.drop('TotalCharges', axis=1, inplace=True)
X_test.drop('TotalCharges', axis=1, inplace=True)  # FIX: mirror drop on X_test

In [ ]:
# Cell 20 — VIF Final (all VIF < 10 ✅)
col_list = []  # FIX: reset collist
for i in X_train.columns:
    if X_train[i].dtype != 'object':
        col_list.append(i)

z = X_train[col_list]
vif_df = pd.DataFrame()
vif_df['Features'] = z.columns
vif_df['VIF'] = [variance_inflation_factor(z.values, i) for i in range(z.shape[1])]
print('Max VIF:', vif_df.VIF.max())  # should be < 10
vif_df

In [ ]:
# Cell 21 — Correlation Heatmap (safe — all numeric)
plt.figure(figsize=(18, 12))
sns.heatmap(X_train.corr(), annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap of Features')
plt.tight_layout()
plt.show()